# A/B R&D: changepoint in mean

**Task (MCP arm):** Synthetic series with one mean shift. Compare **grid SSE** vs **CUSUM-style** estimator; run **center** vs **edge** τ*. Use numpy + matplotlib. Save figures with clear names.

In [ ]:
# setup
import os
import numpy as np
import matplotlib.pyplot as plt


In [2]:
def series(tau, n=400, sigma=0.4, rng_seed=4242):
    rng = np.random.default_rng(rng_seed)
    y = np.zeros(n)
    mu0, mu1 = 0.0, 1.5
    y[:tau] = mu0 + rng.normal(0, sigma, tau)
    y[tau:] = mu1 + rng.normal(0, sigma, n - tau)
    return y
tau_A, tau_B = 200, 45
yA, yB = series(tau_A), series(tau_B)
print('A', yA.shape, yA[:3], yA[199:202])
print('B', yB.shape, yB[43:47])


A (400,) [ 0.18648831 -0.04257061 -1.31410241] [-0.40889092  1.72707136  1.53256973]
B (400,) [0.46206121 0.80988009 0.93603639 0.94469409]


In [3]:
# estimators: SSE grid on interior window; |CUSUM| argmax (exclude edges)
n = 400


def tau_grid_bounds(tau_star, nobs):
    lo = max(1, tau_star - 40)
    hi = min(nobs - 1, tau_star + 60)
    return lo, hi


def sse_profile(y, lo, hi):
    taus = np.arange(lo, hi + 1, dtype=int)
    prof = np.empty_like(taus, dtype=float)
    for j, t in enumerate(taus):
        m0 = y[:t].mean()
        m1 = y[t:].mean()
        prof[j] = ((y[:t] - m0) ** 2).sum() + ((y[t:] - m1) ** 2).sum()
    return taus, prof


def tau_hat_grid(y, tau_star):
    lo, hi = tau_grid_bounds(tau_star, len(y))
    taus, prof = sse_profile(y, lo, hi)
    return int(taus[np.argmin(prof)])


def tau_hat_cusum(y, edge=15):
    mu = y.mean()
    c = np.cumsum(y - mu)
    s, e = edge, len(y) - edge
    idx = s + np.argmax(np.abs(c[s:e]))
    return int(idx)


def abs_err(y, tau_star, method):
    if method == 'grid':
        th = tau_hat_grid(y, tau_star)
    else:
        th = tau_hat_cusum(y)
    return abs(th - tau_star), th


In [4]:
# raw series + annotations; save combined raw plot (prefix from MCP env or fallback)
import os

FIG_PREFIX = os.environ.get('LLM_NB_ART_PREFIX') or 'ab_rnd_cp_noseed_'
demos_dir = 'benchmarks/case_studies'

fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
for ax_i, (yy, ts, lab) in zip(
    ax,
    [
        (yA, tau_A, 'A center τ*=200'),
        (yB, tau_B, 'B edge τ*=45'),
    ],
):
    ax_i.plot(yy, lw=0.8, color='0.2')
    ax_i.axvline(ts - 0.5, color='C3', ls='--', lw=1, label=f'true τ={ts}')
    ax_i.set_ylabel('y')
    ax_i.set_title(lab)
    ax_i.legend(loc='upper right', fontsize=8)
ax[1].set_xlabel('t')
fig.tight_layout()
raw_path = os.path.join(demos_dir, f'{FIG_PREFIX}_raw_AB_twopanel.png')
fig.savefig(raw_path, dpi=150)
plt.close()
print('saved', raw_path)


saved demos\20260503_gen1_f6f34a18_raw_AB_twopanel.png


In [5]:
# error table both scenarios × both methods
rows = []
for tag, y, ts in [('A', yA, tau_A), ('B', yB, tau_B)]:
    for method in ('grid', 'cusum'):
        err, th = abs_err(y, ts, method)
        rows.append((tag, ts, method, th, err))
print('scenario true_tau method tau_hat abs_err')
for r in rows:
    print('%s %s %s %s %s' % r)


scenario true_tau method tau_hat abs_err
A 200 grid 200 0
A 200 cusum 199 1
B 45 grid 44 1
B 45 cusum 46 1


In [6]:
# diagnostic: SSE curves + |CUSUM| for both (second figure)
import os

FIG_PREFIX = os.environ.get('LLM_NB_ART_PREFIX') or 'ab_rnd_cp_noseed_'
demos_dir = 'benchmarks/case_studies'

fig, axes = plt.subplots(2, 2, figsize=(10, 5))
for row, (y, ts, name) in enumerate([(yA, tau_A, 'A'), (yB, tau_B, 'B')]):
    lo, hi = tau_grid_bounds(ts, len(y))
    taus, prof = sse_profile(y, lo, hi)
    axes[row, 0].plot(taus, prof, lw=1)
    axes[row, 0].axvline(ts, color='C3', ls='--', lw=1)
    thg = tau_hat_grid(y, ts)
    axes[row, 0].axvline(thg, color='C0', ls=':', lw=1)
    axes[row, 0].set_title(f'{name} SSE segment')
    axes[row, 0].set_xlabel('candidate τ')
    mu = y.mean()
    c = np.cumsum(y - mu)
    axes[row, 1].plot(np.arange(len(y)), np.abs(c), lw=0.8)
    axes[row, 1].axvline(ts, color='C3', ls='--', lw=1)
    thc = tau_hat_cusum(y)
    axes[row, 1].axvline(thc, color='C2', ls=':', lw=1)
    axes[row, 1].set_title(f'{name} |CUSUM|')
    axes[row, 1].set_xlabel('t')
fig.tight_layout()
diag_path = os.path.join(demos_dir, f'{FIG_PREFIX}_diag_sse_cusum.png')
fig.savefig(diag_path, dpi=150)
plt.close()
print('saved', diag_path)


saved demos\20260503_gen1_f6f34a18_diag_sse_cusum.png
